# Eval test set

Evaluate a 6-second audio model from `app/model_prob/` on the test index `crop_data/test_6.csv`.

Labels follow the training notebooks: `0 = fake`, `1 = real`.

In [1]:
from pathlib import Path
import os
import sys

import pandas as pd
import torch
from safetensors.torch import load_file
from torch.utils.data import DataLoader

# Make the notebook work whether it is launched from the repo root or train_eval/.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "crop_data").exists() and (PROJECT_ROOT.parent / "crop_data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config.loader import get_training_default_variant, get_training_model_kwargs, get_training_variants
from model_audio_input.dataset import SonicDataset
from model_audio_input.model import SpecTTTra

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Project root:", PROJECT_ROOT)
print("Device:", device)

Project root: /media/bonxom/CA8691E68691D2F5/AI_Music_detection
Device: cuda


In [7]:
TEST_CSV = PROJECT_ROOT / "crop_data" / "test_6.csv"
TEST_ROOT = PROJECT_ROOT / "crop_data" / "test"
MODEL_DIR = PROJECT_ROOT / "app" / "model_prob"

# Set to a file name such as "model_alpha_20e.safetensors" to force one model.
# If None, this follows the app-style default: the first model matching the default variant.
MODEL_NAME = "model_alpha_2e.safetensors"

DURATION_SECONDS = 6.0
BATCH_SIZE = 32
NUM_WORKERS = 6
THRESHOLD = 0.5

print("Test CSV:", TEST_CSV)
print("Test root:", TEST_ROOT)
print("Model dir:", MODEL_DIR)

Test CSV: /media/bonxom/CA8691E68691D2F5/AI_Music_detection/crop_data/test_6.csv
Test root: /media/bonxom/CA8691E68691D2F5/AI_Music_detection/crop_data/test
Model dir: /media/bonxom/CA8691E68691D2F5/AI_Music_detection/app/model_prob


In [8]:
def configured_variants() -> set[str]:
    return {
        str(item.get("name", "")).strip()
        for item in get_training_variants()
        if str(item.get("name", "")).strip()
    }


def infer_variant(weights_path: Path) -> str:
    env_variant = os.getenv("AI_MODEL_VARIANT")
    if env_variant and env_variant.strip():
        return env_variant.strip()

    stem = weights_path.stem.lower()
    for variant in sorted(configured_variants()):
        if variant.lower() in stem:
            return variant

    return get_training_default_variant()


def inspect_model_weights(weights_path: Path) -> dict[str, int | str]:
    state_dict = load_file(str(weights_path), device="cpu")
    classifier_weight = state_dict.get("classifier.weight")
    classifier_bias = state_dict.get("classifier.bias")

    if classifier_weight is not None:
        num_classes = int(classifier_weight.shape[0])
    elif classifier_bias is not None:
        num_classes = int(classifier_bias.numel())
    else:
        raise KeyError(f"Unable to determine classifier output size for {weights_path}")

    return {
        "num_classes": num_classes,
        "output_mode": "binary_sigmoid" if num_classes == 1 else "softmax",
    }


def list_model_files(model_dir: Path) -> list[Path]:
    if not model_dir.exists():
        raise FileNotFoundError(f"Model directory not found: {model_dir}")
    files = sorted(model_dir.glob("*.safetensors"), key=lambda p: p.name.lower())
    if not files:
        raise FileNotFoundError(f"No .safetensors models found in {model_dir}")
    return files


def select_model_path(model_dir: Path, model_name: str | None = None) -> Path:
    files = list_model_files(model_dir)

    if model_name:
        selected = model_dir / model_name
        if not selected.exists():
            available = ", ".join(p.name for p in files)
            raise FileNotFoundError(f"Model {model_name!r} not found. Available: {available}")
        return selected

    env_model_name = os.getenv("AI_ACTIVE_MODEL")
    if env_model_name:
        selected = model_dir / env_model_name
        if selected.exists():
            return selected

    default_variant = get_training_default_variant()
    for path in files:
        if infer_variant(path) == default_variant:
            return path

    return files[0]


model_rows = []
for path in list_model_files(MODEL_DIR):
    metadata = inspect_model_weights(path)
    model_rows.append({
        "name": path.name,
        "variant": infer_variant(path),
        "num_classes": metadata["num_classes"],
        "output_mode": metadata["output_mode"],
    })

pd.DataFrame(model_rows)

,name,variant,num_classes,output_mode
0,model_alpha.safetensors,alpha,1,binary_sigmoid
1,model_alpha_20e.safetensors,alpha,1,binary_sigmoid
2,model_alpha_2e.safetensors,alpha,1,binary_sigmoid
3,model_beta_20e.safetensors,beta,1,binary_sigmoid
4,model_gamma_20e.safetensors,gamma,1,binary_sigmoid


In [9]:
def load_eval_model(weights_path: Path, device: torch.device) -> torch.nn.Module:
    metadata = inspect_model_weights(weights_path)
    variant = infer_variant(weights_path)

    model_kwargs = get_training_model_kwargs(variant_name=variant)
    model_kwargs["num_classes"] = int(metadata["num_classes"])
    model_kwargs["expected_samples"] = int(16000 * DURATION_SECONDS)

    model = SpecTTTra(**model_kwargs)
    state_dict = load_file(str(weights_path), device="cpu")
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()

    print("Loaded model:", weights_path)
    print("Variant:", variant)
    print("Output:", metadata)
    return model


weights_path = select_model_path(MODEL_DIR, MODEL_NAME)
model = load_eval_model(weights_path, device)

Loaded model: /media/bonxom/CA8691E68691D2F5/AI_Music_detection/app/model_prob/model_alpha_2e.safetensors
Variant: alpha
Output: {'num_classes': 1, 'output_mode': 'binary_sigmoid'}


In [10]:
def resolve_audio_path(row: pd.Series, project_root: Path, test_root: Path) -> str:
    raw_path = Path(str(row["path"]))
    candidates = []

    if raw_path.is_absolute():
        candidates.append(raw_path)
    else:
        candidates.append(project_root / raw_path)

    candidates.append(test_root / str(row["filename"]))
    label = int(row["label"])
    subdir = "real_6s" if label == 1 else "fake_6s"
    candidates.append(test_root / subdir / str(row["filename"]))

    for candidate in candidates:
        if candidate.exists():
            return str(candidate)

    return str(candidates[0])


def load_test_df(csv_path: Path, test_root: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    required_cols = ["filename", "path", "fake_type", "label"]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    df = df[required_cols].copy()
    df["label"] = df["label"].astype(int)
    df["path"] = df.apply(lambda row: resolve_audio_path(row, PROJECT_ROOT, test_root), axis=1)

    exists_mask = df["path"].map(lambda p: Path(p).exists())
    if exists_mask.sum() < len(df):
        print(f"[WARN] Missing files: {len(df) - exists_mask.sum()} / {len(df)}")

    df = df[exists_mask].reset_index(drop=True)
    if len(df) == 0:
        raise ValueError("No valid audio files found.")

    return df


test_df = load_test_df(TEST_CSV, TEST_ROOT)
print(test_df.shape)
print(test_df["label"].value_counts().sort_index())
test_df.head()

(10235, 4)
label
0    5000
1    5235
Name: count, dtype: int64


,filename,path,fake_type,label
0,real_12326.mp3,/media/bonxom/CA8691E68691D2F5/AI_Music_detect...,Real,1
1,real_12327.mp3,/media/bonxom/CA8691E68691D2F5/AI_Music_detect...,Real,1
2,real_12328.mp3,/media/bonxom/CA8691E68691D2F5/AI_Music_detect...,Real,1
3,real_12329.mp3,/media/bonxom/CA8691E68691D2F5/AI_Music_detect...,Real,1
4,real_12330.mp3,/media/bonxom/CA8691E68691D2F5/AI_Music_detect...,Real,1


In [13]:
def evaluate(
    model,
    df: pd.DataFrame,
    duration_seconds: float = 6.0,
    batch_size: int = 32,
    num_workers: int = 4,
    threshold: float = 0.5,
    device: torch.device | None = None,
):
    test_ds = SonicDataset(df, duration_seconds=duration_seconds)
    test_dl = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model.to(device)
    model.eval()

    all_rows = []
    total = 0
    correct = 0
    tn = fp = fn = tp = 0

    with torch.no_grad():
        for batch in test_dl:
            x = batch["x"].to(device)
            y = batch["y"].to(device).long()

            logits = model(x)
            if logits.ndim == 1:
                logits = logits.unsqueeze(1)

            if logits.ndim == 2 and logits.shape[1] == 1:
                prob_real = torch.sigmoid(logits).squeeze(1)
                pred = (prob_real >= threshold).long()
            elif logits.ndim == 2 and logits.shape[1] == 2:
                probs = torch.softmax(logits, dim=1)
                prob_real = probs[:, 1]
                pred = torch.argmax(probs, dim=1)
            else:
                raise ValueError(f"Unexpected model output shape: {tuple(logits.shape)}")

            total += y.size(0)
            correct += (pred == y).sum().item()

            y_cpu = y.cpu()
            pred_cpu = pred.cpu()
            prob_cpu = prob_real.cpu()

            for i in range(len(y_cpu)):
                label_i = int(y_cpu[i])
                pred_i = int(pred_cpu[i])

                if label_i == 0 and pred_i == 0:
                    tn += 1
                elif label_i == 0 and pred_i == 1:
                    fp += 1
                elif label_i == 1 and pred_i == 0:
                    fn += 1
                elif label_i == 1 and pred_i == 1:
                    tp += 1

                all_rows.append({
                    "filename": batch["filename"][i],
                    "path": batch["path"][i],
                    "fake_type": batch["fake_type"][i],
                    "label": label_i,
                    "pred": pred_i,
                    "prob_real": float(prob_cpu[i]),
                    "correct": label_i == pred_i,
                })

            print(f"Processed {total} samples...", end="\r", flush=True)

    accuracy = correct / total if total > 0 else 0.0
    recall_fake = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    recall_real = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precision_fake = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    precision_real = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1_fake = 2 * precision_fake * recall_fake / (precision_fake + recall_fake) if (precision_fake + recall_fake) > 0 else 0.0
    f1_real = 2 * precision_real * recall_real / (precision_real + recall_real) if (precision_real + recall_real) > 0 else 0.0
    balanced_accuracy = (recall_fake + recall_real) / 2.0

    metrics = {
        "model": weights_path.name,
        "num_samples": total,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "fake_precision": precision_fake,
        "fake_recall": recall_fake,
        "fake_f1": f1_fake,
        "real_precision": precision_real,
        "real_recall": recall_real,
        "real_f1": f1_real,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "confusion_matrix": [[tn, fp], [fn, tp]],
    }

    pred_df = pd.DataFrame(all_rows)

    print("========== Evaluation ==========")
    print(f"Model: {weights_path.name}")
    print(f"Samples: {total}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Balanced Accuracy: {balanced_accuracy:.4f}")
    print()
    print(f"Fake - Precision: {precision_fake:.4f} | Recall: {recall_fake:.4f} | F1: {f1_fake:.4f}")
    print(f"Real - Precision: {precision_real:.4f} | Recall: {recall_real:.4f} | F1: {f1_real:.4f}")
    print()
    print("Confusion Matrix [[TN, FP], [FN, TP]]")
    print([[tn, fp], [fn, tp]])

    return metrics, pred_df

In [14]:
metrics, pred_df = evaluate(
    model,
    test_df,
    duration_seconds=DURATION_SECONDS,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    threshold=THRESHOLD,
    device=device,
)

metrics_df = pd.DataFrame([{k: v for k, v in metrics.items() if k != "confusion_matrix"}])
metrics_df

========== Evaluation ==========
Model: model_alpha_2e.safetensors
Samples: 10235
Accuracy: 0.9930
Balanced Accuracy: 0.9930

Fake - Precision: 0.9914 | Recall: 0.9942 | F1: 0.9928
Real - Precision: 0.9944 | Recall: 0.9918 | F1: 0.9931

Confusion Matrix [[TN, FP], [FN, TP]]
[[4971, 29], [43, 5192]]


,model,num_samples,accuracy,balanced_accuracy,fake_precision,fake_recall,fake_f1,real_precision,real_recall,real_f1,tn,fp,fn,tp
0,model_alpha_2e.safetensors,10235,0.992965,0.992993,0.991424,0.9942,0.99281,0.994446,0.991786,0.993114,4971,29,43,5192


In [15]:
pred_df.head()

,filename,path,fake_type,label,pred,prob_real,correct
0,real_12326.mp3,/media/bonxom/CA8691E68691D2F5/AI_Music_detect...,Real,1,1,0.998244,True
1,real_12327.mp3,/media/bonxom/CA8691E68691D2F5/AI_Music_detect...,Real,1,1,0.996202,True
2,real_12328.mp3,/media/bonxom/CA8691E68691D2F5/AI_Music_detect...,Real,1,1,0.993244,True
3,real_12329.mp3,/media/bonxom/CA8691E68691D2F5/AI_Music_detect...,Real,1,1,0.997612,True
4,real_12330.mp3,/media/bonxom/CA8691E68691D2F5/AI_Music_detect...,Real,1,1,0.998283,True


In [16]:
summary_by_type = (
    pred_df.groupby(["fake_type", "label"], dropna=False)
    .agg(
        samples=("filename", "count"),
        accuracy=("correct", "mean"),
        mean_prob_real=("prob_real", "mean"),
    )
    .reset_index()
)
summary_by_type

,fake_type,label,samples,accuracy,mean_prob_real
0,Real,1,5235,0.991786,0.974488
1,half fake,0,536,1.000000,0.004644
2,mostly fake,0,4464,0.993504,0.015443


In [17]:
PREDICTION_CSV = PROJECT_ROOT / "train_eval" / "eval_test_predictions.csv"
METRICS_CSV = PROJECT_ROOT / "train_eval" / "eval_test_metrics.csv"

pred_df.to_csv(PREDICTION_CSV, index=False)
metrics_df.to_csv(METRICS_CSV, index=False)

print("Saved predictions to:", PREDICTION_CSV)
print("Saved metrics to:", METRICS_CSV)

Saved predictions to: /media/bonxom/CA8691E68691D2F5/AI_Music_detection/train_eval/eval_test_predictions.csv
Saved metrics to: /media/bonxom/CA8691E68691D2F5/AI_Music_detection/train_eval/eval_test_metrics.csv
